# Transfer Learning from ResNet50

In [1]:
import torch, torchvision
import numpy as np
from collect_data import collect_data, train_model

DATA_DIR = '../data/images/climbs'

# Load pretrained ResNet model
resnet50 = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.IMAGENET1K_V1)

# Transform images to batch_size x channels x rows x columns
preprocess = torchvision.transforms.Compose([
    torchvision.transforms.Resize((224, 224)),
    torchvision.transforms.ToTensor(),
    torchvision.transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Get training and validation data
train_loader, val_loader, test_loader = collect_data(DATA_DIR, preprocess)

In [2]:
print(resnet50)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Con

In [3]:
class ResNetTransfer(torch.nn.Module):
    def __init__(self, resnetmodel):
        super(ResNetTransfer, self).__init__()
        # ResNet structure (without classifier)
        self.resnet = torch.nn.Sequential(resnetmodel.conv1,
                                            resnetmodel.bn1,
                                            resnetmodel.relu,
                                            resnetmodel.maxpool,
                                            resnetmodel.layer1,
                                            resnetmodel.layer2,
                                            resnetmodel.layer3,
                                            resnetmodel.layer4,
                                            resnetmodel.avgpool)
        # 2048 from the resnet convolution and 1 from the angles
        self.regressor = torch.nn.Sequential(torch.nn.Linear(2049, 1))
    
    def forward(self, X, angles):
        r = self.resnet(X)
        f = torch.flatten(r, 1)
        f = torch.cat([f, angles], dim=1)
        y = self.regressor(f)
        return y

In [4]:
device = torch.device('cpu') 
if torch.cuda.is_available():
    device = torch.device('cuda') # rose laptops and the servers
elif torch.backends.mps.is_available():
    device = torch.device('mps') # mac
print('Using device:', device)

Using device: cuda


In [5]:
import copy

model = ResNetTransfer(copy.deepcopy(resnet50)).to(device)

# Freeze the pretrained convolutional feature extractor. 
# Only the new Linear classifier gets trained.
for param in model.resnet.parameters():
    param.requires_grad = False

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total trainable parameters: {total_params}')

Total trainable parameters: 2050


In [6]:
# Hyperparameters
epsilon = 1E-4

# Use MSE loss
criterion = torch.nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=epsilon)

train_model(model, train_loader, val_loader, criterion, optimizer, device, num_epochs=100, checkpoint_path='resnet_transfer_regressor.pth')

Epoch [0/100], Train Loss: 60.54879, Val Loss: 55.39708 *, Train Acc: 0.00000, Validation Acc: 0.06667
Epoch [10/100], Train Loss: 37.50082, Val Loss: 35.18571 *, Train Acc: 0.00000, Validation Acc: 0.00000
Epoch [20/100], Train Loss: 16.64967, Val Loss: 20.97571 *, Train Acc: 0.11429, Validation Acc: 0.00000
Epoch [30/100], Train Loss: 12.39460, Val Loss: 14.38100 *, Train Acc: 0.02857, Validation Acc: 0.06667
Epoch [40/100], Train Loss: 14.28184, Val Loss: 11.37788 *, Train Acc: 0.07143, Validation Acc: 0.06667
Epoch [50/100], Train Loss: 13.64548, Val Loss: 10.17340 *, Train Acc: 0.22857, Validation Acc: 0.20000
Epoch [60/100], Train Loss: 9.96892, Val Loss: 9.84496, Train Acc: 0.30000, Validation Acc: 0.20000
Epoch [70/100], Train Loss: 15.12749, Val Loss: 9.62604, Train Acc: 0.12857, Validation Acc: 0.20000
Epoch [80/100], Train Loss: 8.18353, Val Loss: 9.41227, Train Acc: 0.30000, Validation Acc: 0.20000
Epoch [90/100], Train Loss: 8.19050, Val Loss: 9.25983 *, Train Acc: 0.30000

In [7]:
# Test model on test split
from test_model import test_model

test_model(ResNetTransfer(copy.deepcopy(resnet50)).to(device), 'resnet_transfer_regressor.pth', criterion, test_loader, device)

Loss: 7.846875190734863
Accuracy: 0.125


In [8]:
# Fine-tuning
finetune_model = ResNetTransfer(copy.deepcopy(resnet50)).to(device)
finetune_model.load_state_dict(torch.load('resnet_transfer_regressor.pth', map_location=device))

for param in finetune_model.resnet.parameters():
    param.requires_grad = True

# Use different learning rate on different params to train resnet layers more slowly
finetune_optimizer = torch.optim.Adam([
    {'params': finetune_model.resnet.parameters(), 'lr': 1e-6},
    {'params': finetune_model.regressor.parameters(), 'lr': epsilon},
])

train_model(finetune_model, train_loader, val_loader, criterion, finetune_optimizer, device, num_epochs=100, checkpoint_path='resnet_finetuned_regressor.pth')

Epoch [0/100], Train Loss: 5.66921, Val Loss: 9.31296 *, Train Acc: 0.30000, Validation Acc: 0.20000
Epoch [10/100], Train Loss: 6.42813, Val Loss: 8.41125 *, Train Acc: 0.28571, Validation Acc: 0.26667
Epoch [20/100], Train Loss: 3.73829, Val Loss: 7.83894 *, Train Acc: 0.21429, Validation Acc: 0.20000
Epoch [30/100], Train Loss: 3.14369, Val Loss: 7.49390 *, Train Acc: 0.28571, Validation Acc: 0.20000
Epoch [40/100], Train Loss: 1.87539, Val Loss: 7.10649 *, Train Acc: 0.30000, Validation Acc: 0.13333
Epoch [50/100], Train Loss: 2.35109, Val Loss: 6.84516, Train Acc: 0.34286, Validation Acc: 0.26667
Epoch [60/100], Train Loss: 2.01282, Val Loss: 6.55583 *, Train Acc: 0.28571, Validation Acc: 0.20000
Epoch [70/100], Train Loss: 0.39537, Val Loss: 6.34874, Train Acc: 0.60000, Validation Acc: 0.20000
Epoch [80/100], Train Loss: 0.50537, Val Loss: 6.26157, Train Acc: 0.52857, Validation Acc: 0.13333
Epoch [90/100], Train Loss: 0.41741, Val Loss: 5.95148, Train Acc: 0.65714, Validation Ac

In [9]:
# Test model on test split
from test_model import test_model

test_model(ResNetTransfer(copy.deepcopy(resnet50)).to(device), 'resnet_finetuned_regressor.pth', criterion, test_loader, device)

Loss: 6.22596549987793
Accuracy: 0.125
